In [0]:
import pandas as pd


df=pd.read_csv('source_data.csv')
spark_df=spark.createDataFrame(df)
spark_df.write.mode("overwrite").saveAsTable("bronze_table_raw_data")

df_mismatch_type=df.copy()
numeric_cols = ['rpm_max', 'oph', 'pist_m', 'ng_imp', 'bmep']
for col in numeric_cols:
    df_mismatch_type[col] = pd.to_numeric(df_mismatch_type[col], errors="coerce")
df_mismatch_type = df_mismatch_type[df_mismatch_type['issue_type'].map(type) == str]
df_mismatch_type['past_dmg'] = df_mismatch_type['past_dmg'].map({1: True, 0: False})
df_mismatch_type=df_mismatch_type.dropna(thresh=14)
df_rename=df_mismatch_type.copy()
df_rename=df_mismatch_type.rename(columns={'oph':'operating_hours','bmep':'break_mean_effective_pressure'})
spark_df_silver=spark.createDataFrame(df_rename)
spark_df_silver.write.saveAsTable("silver_table_clean_data")
df_silver=df_rename.copy()
df_silver=df_rename[(df_rename['issue_type'] != 'ddddd') & (df_rename['issue_type'] != 'non-typical')]
spark_df_silver_final=spark.createDataFrame(df_silver)
spark_df_silver_final.write.mode("overwrite").saveAsTable("silver_table_clean_data")


df_noduplicates=df_silver.drop_duplicates()
df_gold=df_noduplicates.copy()
df_gold=df_noduplicates[(df_noduplicates['operating_hours'] <= 120000 )]
spark_df_gold=spark.createDataFrame(df_gold)
spark_df_gold.write.mode("overwrite").saveAsTable("gold_table_business_tests")




